# Week 1 · Day 3 — Lab 4
## Transformation: `assign`, vectorized ops, `groupby`, `merge`, reshape

> **AI Engineering Academy** · Gamut Technology Services · pandas 3.x

This is where raw tables become features. You'll derive columns immutably with
**`assign`** (including pandas 3.x's `pd.col()` syntax), do work at C-speed with the
**`.str`** and **`.dt`** accessors and **`np.where`/`pd.cut`** instead of slow
`apply`, summarize with **`groupby.agg`/`transform`**, combine tables with a
validated **`merge`**, and reshape between wide and long with **`pivot_table`/`melt`**.

### Learning objectives
1. Add derived columns with `assign` (lambda and `pd.col()`), understanding it never mutates.
2. Replace `apply` with vectorized `.str`, `.dt`, `np.where`, and `pd.cut`.
3. Aggregate with `groupby().agg(named=...)` and broadcast group results back with `transform`.
4. Combine tables with `merge`, using `validate=` to catch bad cardinality.
5. Reshape wide↔long with `pivot_table` and `melt`.

### Time budget — ~75 min
| Segment | Time |
|---|---|
| Framing & objectives | 5 min |
| **A.** `assign` | 12 min |
| **B.** Vectorized ops (`.str`, `.dt`, `np.where`, `pd.cut`) | 16 min |
| **C.** `groupby` + `transform` | 16 min |
| **D.** `merge` with `validate` | 14 min |
| **E.** Reshape: `pivot_table` / `melt` | 10 min |
| Wrap-up + stretch | 2 min |

### Files you need (in `data/`)
- `users.csv` — 2000 users.
- `events.csv` — 8000 events.
- `monthly_tokens_long.csv` — 72 rows (12 users × 6 months), long form, for reshaping.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print("pandas", pd.__version__)   # target: pandas 3.x on Python 3.13

# Solution is different here because of folder structure

DATA = Path("../data")

def check(label, predicate):
    try:
        ok = bool(predicate())
    except Exception as exc:
        ok = False
        label = f"{label}  (raised {type(exc).__name__}: {exc})"
    print(("PASS " if ok else "FAIL "), label)
    return ok

users = pd.read_csv(
    DATA / "users.csv",
    dtype={"user_id": "int32", "plan": "category", "region": "category"},
    parse_dates=["signup_date"],
)
events = pd.read_csv(
    DATA / "events.csv",
    dtype={"user_id": "int32", "model": "category", "score": "float32"},
    parse_dates=["event_date"],
)
print("users:", users.shape, "| events:", events.shape)

## Part A — `assign`: derive columns immutably  *(guided)*

`assign` returns a **new** DataFrame with added/overwritten columns — it never
mutates the caller. Lambdas let a later column reference one derived earlier in the
same call. pandas 3.x also adds **`pd.col()`** expression syntax as a lambda-free
alternative.


In [ ]:
out = events.assign(
    score_pct=lambda x: x["score"] / 100,
    is_top=lambda x: x["score"] >= 90,
)
print(out[["score", "score_pct", "is_top"]].head(3))
print("original untouched (no score_pct):", "score_pct" not in events.columns)

### Exercise A1 — Two derived columns, two syntaxes
Build `enriched` from `events` with:
- `cost_usd` via **lambda**: `input_tokens * 3e-6 + output_tokens * 6e-6`.
- `score_scaled` via **`pd.col()`**: `score` divided by the max score.
Confirm `events` itself gained no new columns (assign is pure).


In [ ]:
enriched = events.assign(
    cost_usd=lambda x: x["input_tokens"] * 3e-6 + x["output_tokens"] * 6e-6,
    score_scaled=pd.col("score") / pd.col("score").max(),
)
mutated = "cost_usd" in events.columns
print(enriched[["input_tokens", "output_tokens", "cost_usd", "score_scaled"]].head(3))
print("events mutated?", mutated)

In [ ]:
check("A1: enriched has both new columns",
      lambda: {"cost_usd", "score_scaled"} <= set(enriched.columns))
check("A1: cost_usd computed correctly for row 0",
      lambda: np.isclose(enriched["cost_usd"].iloc[0],
                         events["input_tokens"].iloc[0]*3e-6 + events["output_tokens"].iloc[0]*6e-6))
check("A1: score_scaled max is 1.0",
      lambda: np.isclose(enriched["score_scaled"].max(), 1.0))
check("A1: assign did NOT mutate events", lambda: mutated is False)

🧑‍🏫 **Instructor note — A1.** The purity of `assign` is the point — no accidental
mutation, chain-friendly. `pd.col()` is genuinely new in pandas 3.x: it lets you
write column expressions without a lambda (`pd.col("score")/pd.col("score").max()`).
Both styles work; lambdas are still needed when a step references a column derived
earlier in the *same* assign call.


## Part B — Vectorized operations (beat `apply`)

Vectorized ops run at C speed and avoid Python-loop overhead. The `.str` accessor
does string ops; `.dt` does datetime ops; `np.where` is a vectorized if/else;
`pd.cut` bins a numeric column into labeled bands. Reach for these **before** ever
writing `apply`.


In [ ]:
# .dt accessor
yr = events["event_date"].dt.year
print("years present:", sorted(yr.unique().tolist()))

# np.where — vectorized if/else
lab = np.where(events["score"] >= 60, "pass", "fail")
print("pass/fail:", pd.Series(lab).value_counts().to_dict())

### Exercise B1 — Model family via `.str`
Derive `model_family`: the part of `model` before the `-` (so `atlas-pro` →
`atlas`). **Cast the categorical `model` to `str` first**, then use
`.str.split("-").str[0]`. Capture the value counts into `family_counts`.


In [ ]:
model_family = events["model"].astype("str").str.split("-").str[0]
family_counts = model_family.value_counts()
print(family_counts)

In [ ]:
check("B1: families are atlas / nova / orion",
      lambda: set(family_counts.index) == {"atlas", "nova", "orion"})
check("B1: atlas is the largest family (mini+pro)",
      lambda: family_counts.idxmax() == "atlas" and int(family_counts["atlas"]) == 3194 + 1986)

### Exercise B2 — Score bands via `pd.cut`
Bin `score` into three labeled bands with `pd.cut`, bins `[0, 60, 80, 100]`,
labels `["low", "mid", "high"]`, into `band`. Then `band_counts` = its value
counts. (This replaces a slow `apply` with a single vectorized call.)


In [ ]:
band = pd.cut(events["score"], bins=[0, 60, 80, 100], labels=["low", "mid", "high"])
band_counts = band.value_counts()
print(band_counts)

In [ ]:
check("B2: three bands present",
      lambda: set(band_counts.index) == {"low", "mid", "high"})
check("B2: bands cover all non-null scores",
      lambda: int(band_counts.sum()) == int(events["score"].notna().sum()))

🧑‍🏫 **Instructor note — B.** The category `.str` gotcha (B1) is worth 60 seconds:
`.str` on a `category` column operates oddly — cast to `str` first. Then hammer the
performance framing: `np.where`, `.str`, `.dt`, and `pd.cut` are C-speed; the same
logic in `apply(..., axis=1)` is a Python loop that can be 100× slower on large
frames. "Search for a vectorized path first" is the rule.


## Part C — `groupby` and `transform`

`groupby().agg(named=(col, func))` produces one row per group with clearly-named
aggregate columns. `transform` is the feature-engineering workhorse: it returns a
result **aligned to the original rows**, so you can attach a group statistic to
every row without a merge.


In [ ]:
by_model = events.groupby("model", observed=True).agg(
    avg_score=("score", "mean"),
    max_score=("score", "max"),
    n=("event_id", "count"),
)
print(by_model.round(2))

### Exercise C1 — Named aggregation per model
Group `events` by `model` (use `observed=True`) and produce `stats` with three
named columns: `avg_score` = mean score, `total_input` = sum of `input_tokens`,
`n_events` = count of `event_id`. It should have one row per model (4 rows).


In [ ]:
stats = events.groupby("model", observed=True).agg(
    avg_score=("score", "mean"),
    total_input=("input_tokens", "sum"),
    n_events=("event_id", "count"),
)
print(stats.round(2))

In [ ]:
check("C1: one row per model", lambda: len(stats) == 4)
check("C1: expected named columns",
      lambda: set(stats.columns) == {"avg_score", "total_input", "n_events"})
check("C1: event counts sum to 8000",
      lambda: int(stats["n_events"].sum()) == 8000)

### Exercise C2 — Attach the group mean with `transform`
Add a column `model_avg_score` to a copy of `events` (call it `feat`) that holds,
for every row, the **mean score of that row's model** — using
`groupby(...).transform("mean")`. The result must have the same length as `events`
(that's what `transform` guarantees).


In [ ]:
feat = events.copy()
feat["model_avg_score"] = feat.groupby("model", observed=True)["score"].transform("mean")
print(feat[["model", "score", "model_avg_score"]].head(3).round(2))

In [ ]:
check("C2: model_avg_score column added",
      lambda: "model_avg_score" in feat.columns)
check("C2: transform preserved row count",
      lambda: len(feat) == len(events))
check("C2: each row's value equals its model's mean",
      lambda: np.isclose(
          feat.loc[feat["model"] == "nova-4", "model_avg_score"].iloc[0],
          events.loc[events["model"] == "nova-4", "score"].mean()))

🧑‍🏫 **Instructor note — C.** The `agg` vs `transform` distinction is the keeper:
`agg` collapses to one row per group; `transform` keeps every original row and
broadcasts the group stat back. `transform` is how you build features like
"deviation from the group mean" without losing rows or doing a merge. Always pass
`observed=True` with categorical groupers in pandas 3.x.


## Part D — `merge` with `validate`

`merge` is the general-purpose join. Be explicit: name the key with `on=`, the
direction with `how=`, and — critically — the expected cardinality with
`validate=`. A silent many-to-many join can multiply rows and corrupt every
downstream aggregate; `validate` turns that into an immediate error.


In [ ]:
# one user -> many events, so users:events is 1:many
merged = users.merge(events, on="user_id", how="left", validate="1:m")
print("merged rows:", len(merged), "(>= 8000 events)")
print("columns:", len(merged.columns))

### Exercise D1 — Join and derive `days_active`
Left-join `users` to `events` on `user_id` with `validate="1:m"` into `joined`.
Then add `days_active` = `(event_date - signup_date).dt.days` via `assign`. Confirm
`days_active` is never negative (events always occur on/after signup).


In [ ]:
joined = (
    users.merge(events, on="user_id", how="left", validate="1:m")
    .assign(days_active=lambda x: (x["event_date"] - x["signup_date"]).dt.days)
)
min_days = joined["days_active"].min()
print("joined shape:", joined.shape)
print("min days_active:", min_days)

In [ ]:
check("D1: join kept at least all 8000 events",
      lambda: len(joined) >= 8000)
check("D1: days_active column exists",
      lambda: "days_active" in joined.columns)
check("D1: days_active is never negative",
      lambda: float(min_days) >= 0)

🧑‍🏫 **Instructor note — D1.** `validate="1:m"` is the habit to instill: it asserts
each left-side user maps to many events and raises if that's violated (e.g. a
duplicated user_id would make it m:m and silently multiply rows). The row count is
8031 — the 8000 events plus one all-NaN row for each of the 31 users who have no
events (that's the `how="left"` behavior). Subtracting datetimes yields a timedelta;
`.dt.days` pulls whole days.


## Part E — Reshape: `pivot_table` and `melt`

**Wide** (one column per category) and **long/tidy** (one row per observation) are
two shapes of the same data. `pivot_table` goes long→wide (and aggregates
duplicates safely); `melt` goes wide→long.


In [ ]:
long = pd.read_csv(DATA / "monthly_tokens_long.csv")
print("long form (tidy):", long.shape)
long.head(3)

### Exercise E1 — Long → wide → long
1. Pivot `long` to wide with `pivot_table`: `index="user_id"`, `columns="month"`,
   `values="tokens"`, `aggfunc="sum"`, `fill_value=0` → `wide`.
2. Melt it back with `reset_index().melt(...)`: `id_vars="user_id"`,
   `var_name="month"`, `value_name="tokens"` → `back_to_long`.
Confirm `wide` is 12×6 and the round-trip preserves the total token sum.


In [ ]:
wide = long.pivot_table(index="user_id", columns="month",
                        values="tokens", aggfunc="sum", fill_value=0)
back_to_long = wide.reset_index().melt(id_vars="user_id",
                                       var_name="month", value_name="tokens")
print("wide shape:", wide.shape)
print("back_to_long shape:", back_to_long.shape)

In [ ]:
check("E1: wide is 12 users x 6 months", lambda: wide.shape == (12, 6))
check("E1: melt returns to long form (72 rows)",
      lambda: len(back_to_long) == 72)
check("E1: round-trip preserves total tokens",
      lambda: int(back_to_long["tokens"].sum()) == int(long["tokens"].sum()))

🧑‍🏫 **Instructor note — E1.** Use `pivot_table` (not bare `pivot`) whenever
duplicate index/column pairs are possible — `pivot` raises on duplicates,
`pivot_table` aggregates them. `melt` is the inverse. Frame it: models and ML
libraries want *long/tidy* input; reports and humans want *wide*. Reshaping between
them is an everyday move.


## Stretch goals *(for fast finishers)*

**S1 — Filter groups by size.** From `events`, keep only rows whose `model` group
has at least 1500 events, using `groupby(...).filter(lambda g: len(g) >= 1500)` into
`big_models`. Which models survive?

**S2 — `map` for a lookup.** Map each `model` to a coarse tier with
`.map({"atlas-mini": "cheap", "atlas-pro": "mid", "nova-4": "mid", "orion-8b":
"premium"})` into a Series `tier`, and count the tiers.


In [ ]:
big_models = events.groupby("model", observed=True).filter(lambda g: len(g) >= 1500)
surviving = set(big_models["model"].unique())
print("surviving models:", surviving)

mapping = {"atlas-mini": "cheap", "atlas-pro": "mid", "nova-4": "mid", "orion-8b": "premium"}
tier = events["model"].astype("str").map(mapping)
tier_counts = tier.value_counts()
print(tier_counts.to_dict())

In [ ]:
check("S1: only models with >= 1500 events remain",
      lambda: surviving == {"atlas-mini", "atlas-pro", "nova-4"})
check("S2: tiers mapped for every row",
      lambda: int(tier_counts.sum()) == len(events))

## Wrap-up — what you can now do

- Derive columns immutably with `assign` (lambda and `pd.col()`).
- Replace `apply` with vectorized `.str`, `.dt`, `np.where`, and `pd.cut`.
- Aggregate with `groupby().agg(named=...)` and broadcast group stats with `transform`.
- Combine tables with a `validate=`d `merge` and derive cross-column features.
- Reshape wide↔long with `pivot_table` and `melt`.

**Next:** Lab 5 — method chaining, `.pipe()`, the three big pitfalls, and an
end-to-end pipeline that ties the whole day together.
